# Train Saylient AI — your own politeness LLM (Qwen2.5-1.5B + QLoRA)

Fine-tune **Qwen/Qwen2.5-1.5B-Instruct** on the Saylient v2 dataset (**49,995 examples**: 500 hand-written seeds + deterministic expansion, 11 intents, Brown & Levinson strategies) using QLoRA on a **free Colab T4 GPU**.

**Before you start:**
- In Colab, set Runtime → Change runtime type → **T4 GPU**.
- This notebook clones `arsalanahmedk/saylient_ai` (public repo) — no API keys needed.
- Training takes roughly 1.5–3 hours on a T4 for 3 epochs over 42,493 train rows. The LoRA adapter (a few hundred MB) is saved to your Google Drive.
- Never commit the adapter or any weights to the repo — Drive only.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Mount Google Drive (the adapter will be saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/saylient-ai', exist_ok=True)
print('Drive mounted.')

## 2. Install training dependencies (Unsloth + TRL + PEFT + bitsandbytes)

Takes a few minutes. Restart the runtime if Colab asks, then re-run from the mount cell.

In [ ]:
!pip install -q unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
print('installed')

## 3. Get the code + v2 dataset

Clones the public repo `arsalanahmedk/saylient_ai` (underscore). Dataset files used: `ml/data/splits_v2/train.jsonl` (42,493 rows), `ml/data/splits_v2/valid.jsonl` (3,751 rows). Training script: `ml/training/train_qlora_v2.py` (base model Qwen/Qwen2.5-1.5B-Instruct).

In [ ]:
!rm -rf /content/saylient_ai
!git clone -q https://github.com/arsalanahmedk/saylient_ai.git /content/saylient_ai
%cd /content/saylient_ai
!ls ml/data/splits_v2/ && wc -l ml/data/splits_v2/*.jsonl ml/data/seeds_v2.jsonl

## 4. Sanity-check the dataset (quick, CPU only)

In [ ]:
!python3 -c "
import json
rows=[json.loads(l) for l in open('ml/data/splits_v2/train.jsonl')]
print('train rows:', len(rows))
print('example:', json.dumps(rows[0], indent=2)[:800])
"
!python3 -m ml.evaluation.eval_v2 --limit 200 --samples 2

## 5. Train (QLoRA, Qwen2.5-1.5B-Instruct, 3 epochs)

Saves **only the LoRA adapter** to `/content/drive/MyDrive/saylient-ai/qwen15b-qlora-v2`. If Colab disconnects, re-run from cell 1 and training restarts (no checkpoint resume in this version — keep the tab open).

In [ ]:
!python3 ml/training/train_qlora_v2.py --train ml/data/splits_v2/train.jsonl --val ml/data/splits_v2/valid.jsonl --output_dir /content/drive/MyDrive/saylient-ai/qwen15b-qlora-v2

## 6. Verify the saved adapter on Drive

In [ ]:
!ls -lh /content/drive/MyDrive/saylient-ai/qwen15b-qlora-v2/
!du -sh /content/drive/MyDrive/saylient-ai/qwen15b-qlora-v2/

## 7. Next steps (after training)

1. **Evaluate**: generate predictions on `ml/data/splits_v2/test.jsonl` (3,751 rows, never trained on) with the merged model, then run `python3 -m ml.evaluation.eval_v2 --test ml/data/splits_v2/test.jsonl --predictions predictions.jsonl`. Targets to beat: keyword baseline intent accuracy 0.464, strategy accuracy 0.490, JSON validity 1.0.
2. **Merge + quantise** the adapter with the 1.5B base (see `ml/serving/` in a later phase) and serve it behind the internal inference service — never expose the model server publicly.
3. Do **not** commit the adapter or weights to the repo.